# Reorganization Sanity Test
Verifies that the directory restructuring did not break imports, path resolution, or core logic.  
**No data is written to disk. No network requests are made. No browser is launched.**

In [1]:
import sys, os
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))  # scrape_code/
sys.path.insert(0, os.path.join(_ROOT, 'scrapers'))
sys.path.insert(0, os.path.join(_ROOT, 'utils'))
print('scrape_code root:', _ROOT)
print('scrapers on path:', os.path.join(_ROOT, 'scrapers'))
print('utils on path:   ', os.path.join(_ROOT, 'utils'))

scrape_code root: /home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code
scrapers on path: /home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/scrapers
utils on path:    /home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/utils


## 1. Import Tests

In [2]:
# ── util_funcs ──
try:
    from util_funcs import get_proportion_values, slice_poss_terr_func
    print('PASS  util_funcs imported')
except Exception as e:
    print('FAIL  util_funcs:', e)

# ── util_data_funcs ──
try:
    from util_data_funcs import (
        gather_dataframes, data_combine, data_combine_temp_inter,
        fill_pos, calc_custom_metrics, get_n_top_not, get_min_max_lims,
        _DATA_DIR
    )
    print('PASS  util_data_funcs imported')
except Exception as e:
    print('FAIL  util_data_funcs:', e)

# ── scraper ──
try:
    from scraper import scrape_model
    print('PASS  scraper imported')
except Exception as e:
    print('FAIL  scraper:', e)

PASS  util_funcs imported
PASS  util_data_funcs imported
PASS  scraper imported


## 2. Path Resolution Test
Confirms that `_DATA_DIR` in `util_data_funcs` resolves to the correct `formed_data/` directory regardless of where the notebook is run from.

In [3]:
print('_DATA_DIR resolves to:', os.path.abspath(_DATA_DIR))

expected_subdirs = ['game_data_v2', 'player_data_mk2', 'team_data', 'game_schedule_data', 'archive']
actual_contents = os.listdir(_DATA_DIR)

for d in expected_subdirs:
    full = os.path.join(_DATA_DIR, d)
    status = 'PASS' if os.path.isdir(full) else 'FAIL'
    print(f'{status}  formed_data/{d} exists')

_DATA_DIR resolves to: /home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/formed_data
PASS  formed_data/game_data_v2 exists
PASS  formed_data/player_data_mk2 exists
PASS  formed_data/team_data exists
PASS  formed_data/game_schedule_data exists
PASS  formed_data/archive exists


## 3. `util_funcs` Logic Tests
Tests parsing functions against known ESPN stat string formats.

In [4]:
# get_proportion_values: parses strings like '8 / 10 (80%)'
sample = '8 / 10 (80%)'
tests = [
    ('numerator   via /', get_proportion_values(sample, '/'), 8),
    ('denominator via (', get_proportion_values(sample, '('), 10),
    ('percentage  via )', get_proportion_values(sample, ')'), 0.80),
]
for label, result, expected in tests:
    status = 'PASS' if result == expected else f'FAIL (got {result}, expected {expected})'
    print(f'{status}  get_proportion_values {label}')

# slice_poss_terr_func: parses strings like '45%/55%'
poss_sample = '45%/55%'
h1, h2 = slice_poss_terr_func(poss_sample)
print('PASS' if h1 == 0.45 else f'FAIL (got {h1})', ' slice_poss_terr_func 1st half')
print('PASS' if h2 == 0.55 else f'FAIL (got {h2})', ' slice_poss_terr_func 2nd half')

PASS  get_proportion_values numerator   via /
PASS  get_proportion_values denominator via (
PASS  get_proportion_values percentage  via )
PASS  slice_poss_terr_func 1st half
PASS  slice_poss_terr_func 2nd half


In [5]:
tests

[('numerator   via /', 8, 8),
 ('denominator via (', 10, 10),
 ('percentage  via )', 0.8, 0.8)]

## 4. `gather_dataframes` — CSV Loading Test
Loads one league end-to-end through `gather_dataframes` to confirm path resolution and CSV reading work.

In [6]:
import pandas as pd

test_league = 'URC'
try:
    game_df, player_df, team_df = gather_dataframes(test_league)
    print(f'PASS  gather_dataframes("{test_league}")')
    print(f'      game_df:   {game_df.shape}')
    print(f'      player_df: {player_df.shape}')
    print(f'      team_df:   {team_df.shape}')
except Exception as e:
    print(f'FAIL  gather_dataframes("{test_league}"):', e)

PASS  gather_dataframes("URC")
      game_df:   (1756, 96)
      player_df: (9243, 30)
      team_df:   (73, 6)


/home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/utils/util_data_funcs.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grab_files = lambda league, order: pd.concat(list(map(pd.read_csv, league_dict[league][order])))


In [7]:
# Test a second league that uses data_combine_temp_inter
test_league_2 = 'SixNat'
try:
    g2, p2, t2 = gather_dataframes(test_league_2)
    print(f'PASS  gather_dataframes("{test_league_2}")')
    print(f'      game_df:   {g2.shape}')
    print(f'      player_df: {p2.shape}')
    print(f'      team_df:   {t2.shape}')
except Exception as e:
    print(f'FAIL  gather_dataframes("{test_league_2}"):', e)

PASS  gather_dataframes("SixNat")
      game_df:   (210, 96)
      player_df: (5500, 29)
      team_df:   (36, 5)


## 5. `util_data_funcs` Helper Functions Test
Tests `fill_pos`, `calc_custom_metrics`, `get_n_top_not`, and `get_min_max_lims` on the loaded data.

In [8]:
import numpy as np

# fill_pos — adds off_position and pos_place columns
try:
    player_filled = fill_pos(player_df.copy(), test_league)
    assert 'off_position' in player_filled.columns, 'off_position column missing'
    assert 'pos_place' in player_filled.columns, 'pos_place column missing'
    assert 'league' in player_filled.columns, 'league column missing'
    assert player_filled['league'].iloc[0] == test_league
    print('PASS  fill_pos:', player_filled[['position', 'off_position', 'pos_place']].drop_duplicates().head(3).to_string())
except Exception as e:
    print('FAIL  fill_pos:', e)

# calc_custom_metrics — adds meters_per_passes, adj_meters_per_passes, passes_per_runs
try:
    player_metrics = calc_custom_metrics(player_filled.copy())
    new_cols = ['meters_per_passes', 'adj_meters_per_passes', 'passes_per_runs']
    missing = [c for c in new_cols if c not in player_metrics.columns]
    if missing:
        print('FAIL  calc_custom_metrics: missing columns:', missing)
    else:
        print('PASS  calc_custom_metrics — new columns present, no NaNs:', player_metrics[new_cols].isnull().any().to_dict())
except Exception as e:
    print('FAIL  calc_custom_metrics:', e)

PASS  fill_pos:   position off_position pos_place
0       FB           FB      back
1        W            W      back
2        C            C      back
PASS  calc_custom_metrics — new columns present, no NaNs: {'meters_per_passes': False, 'adj_meters_per_passes': False, 'passes_per_runs': False}


/home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/utils/util_data_funcs.py:193: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['passes_per_runs'].fillna(0.0, inplace=True)
/home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/utils/util_data_funcs.py:194: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work 

In [9]:
# get_n_top_not — classifies rank as top_5 / mid_table / bot_5
try:
    sample_rows = [
        {'rank': 1,  'max_teams': 16, 'expected': 'top_5'},
        {'rank': 8,  'max_teams': 16, 'expected': 'mid_table'},
        {'rank': 15, 'max_teams': 16, 'expected': 'bot_5'},
    ]
    for row in sample_rows:
        result = get_n_top_not(row)
        status = 'PASS' if result == row['expected'] else f'FAIL (got {result})'
        print(f'{status}  get_n_top_not rank={row["rank"]} → {result}')
except Exception as e:
    print('FAIL  get_n_top_not:', e)

# get_min_max_lims — returns (min-std, max+std)
try:
    import pandas as pd
    col = pd.Series([1, 2, 3, 4, 5])
    mn, mx = get_min_max_lims(col)
    assert mn < col.min(), 'min_lim should be below min'
    assert mx > col.max(), 'max_lim should be above max'
    print(f'PASS  get_min_max_lims → ({mn:.2f}, {mx:.2f})')
except Exception as e:
    print('FAIL  get_min_max_lims:', e)

PASS  get_n_top_not rank=1 → top_5
PASS  get_n_top_not rank=8 → mid_table
PASS  get_n_top_not rank=15 → bot_5
PASS  get_min_max_lims → (-0.58, 6.58)


## 6. `scrape_model` Instantiation Test
Confirms the class can be constructed without launching a browser or making any network calls.

In [10]:
try:
    model = scrape_model(
        league_set=[270557],
        season_set=[2024],
        date_set=None,
        update_type='single team',
        driver_type='backend'
    )
    assert hasattr(model, 'league_set')
    assert hasattr(model, 'headers')
    print('PASS  scrape_model instantiation')
    print('      league_set:', model.league_set)
    print('      season_set:', model.season_set)
    print('      driver_type:', model.driver_type)
except Exception as e:
    print('FAIL  scrape_model instantiation:', e)

PASS  scrape_model instantiation
      league_set: [270557]
      season_set: [2024]
      driver_type: backend


## 7. `scrape_model` Internal Parsing Methods Test
Tests the pure-Python parsing methods using mock/sample data — no network or browser required.

In [11]:
# slice_prop_func — same logic as util_funcs but as a method
try:
    sample = '8/10 (80%)'
    assert model.slice_prop_func(sample, '/') == 8,    'numerator mismatch'
    assert model.slice_prop_func(sample, '(') == 10,   'denominator mismatch'
    assert model.slice_prop_func(sample, ')') == 0.80, 'percentage mismatch'
    print('PASS  scrape_model.slice_prop_func')
except Exception as e:
    print('FAIL  scrape_model.slice_prop_func:', e)

# slice_poss_terr_func (method version)
try:
    h1, h2 = model.slice_poss_terr_func('45%/55%')
    assert h1 == 0.45 and h2 == 0.55
    print('PASS  scrape_model.slice_poss_terr_func')
except Exception as e:
    print('FAIL  scrape_model.slice_poss_terr_func:', e)

PASS  scrape_model.slice_prop_func
PASS  scrape_model.slice_poss_terr_func


In [12]:
from bs4 import BeautifulSoup

# label_table_parse — strips label text from a table and pairs remaining rows
try:
    html = '''
    <table><tbody>
      <tr><td>Tries</td><td>3</td><td>2</td></tr>
      <tr><td>Penalties</td><td>4</td><td>5</td></tr>
    </tbody></table>
    '''
    soup = BeautifulSoup(html, 'html.parser')
    cells = soup.find('tbody').find_all('td')
    labels = ['Tries', 'Penalties']
    result = model.label_table_parse(labels, cells)
    assert result == [['3', '2'], ['4', '5']], f'unexpected result: {result}'
    print('PASS  scrape_model.label_table_parse')
    print('      parsed pairs:', result)
except Exception as e:
    print('FAIL  scrape_model.label_table_parse:', e)

PASS  scrape_model.label_table_parse
      parsed pairs: [['3', '2'], ['4', '5']]


In [13]:
# clean_match_stats — tests the DataFrame cleaning pipeline on a minimal synthetic row
try:
    sample_row = {
        'game_id': 'test_001', 'league_id': '270557',
        'home_kick_percent': '75%', 'away_kick_percent': '60%',
        'home_possession_1h_2h': '55%/45%', 'away_possession_1h_2h': '45%/55%',
        'home_territory_1h_2h': '52%/48%', 'away_territory_1h_2h': '48%/52%',
        'home_rucks_won': '35 / 40 (88%)', 'away_rucks_won': '28 / 33 (85%)',
        'home_mauls_won': '3 / 4 (75%)',   'away_mauls_won': '2 / 3 (67%)',
        'home_scrum': '8 / 9 (89%)',       'away_scrum': '7 / 8 (88%)',
        'home_lineout': '10 / 12 (83%)',   'away_lineout': '9 / 11 (82%)',
    }
    df_in = pd.DataFrame([sample_row])
    df_out = model.clean_match_stats(df_in.copy())

    checks = [
        ('home_kick_percent', 0.75),
        ('away_kick_percent', 0.60),
        ('home_1h_poss', 0.55),
        ('away_2h_poss', 0.55),
        ('home_rucks_won_num', 35),
        ('home_rucks_won_total_num', 40),
        ('home_rucks_won_percent', 0.88),
    ]
    all_pass = True
    for col, expected in checks:
        val = df_out[col].iloc[0]
        ok = abs(val - expected) < 0.01
        if not ok:
            print(f'FAIL  clean_match_stats [{col}]: got {val}, expected {expected}')
            all_pass = False
    if all_pass:
        print('PASS  scrape_model.clean_match_stats — all column checks passed')
except Exception as e:
    print('FAIL  scrape_model.clean_match_stats:', e)

PASS  scrape_model.clean_match_stats — all column checks passed


## 8. `_DATA_DIR` Write Path Test (scraper.py)
Confirms the paths that `scraper.py` would write to resolve correctly — without actually writing anything.

In [14]:
import importlib, scraper as scraper_mod

scraper_data_dir = os.path.abspath(scraper_mod._DATA_DIR)
print('scraper._DATA_DIR resolves to:', scraper_data_dir)

write_paths = [
    os.path.join(scraper_data_dir, 'game_schedule_data'),
    os.path.join(scraper_data_dir, 'game_data'),
    os.path.join(scraper_data_dir, 'player_data_mk2'),
]
for p in write_paths:
    status = 'PASS' if os.path.isdir(p) else 'FAIL (dir not found)'
    print(f'{status}  write target exists: {os.path.basename(p)}/')

scraper._DATA_DIR resolves to: /home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/formed_data
PASS  write target exists: game_schedule_data/
FAIL (dir not found)  write target exists: game_data/
PASS  write target exists: player_data_mk2/


## Summary
All cells above should show `PASS`. Any `FAIL` indicates a broken import or path that needs attention.